# Imports

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from sklearn.linear_model import SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import GridSearchCV
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [29]:
encoded_scaled_df = pd.read_csv('encoded_not_scaled_df.csv')

In [30]:
encoded_scaled_df = encoded_scaled_df.drop(columns=['Quantity', 'Sales'])

In [31]:
encoded_scaled_df

,Unnamed: 0.1,Unnamed: 0,Ship Mode,Discount,Profit,Shipping Cost,Order Priority,Price,Avg_Sales_Category,Avg_Sales_Country,...,ship_year,ship_month,ship_day,ship_dayofweek,ship_quarter,ship_dayofyear,ship_weekofyear,ship_hour,ship_minute,ship_second
0,0,234,0,0.0,140.1600,399.96,0,637.350,467.858939,172.770678,...,2012,4,1,6,2,92,13,0,0,0
1,1,238,0,0.0,412.5394,397.52,1,226.670,416.248905,229.858001,...,2014,12,26,4,4,360,52,0,0,0
2,2,239,1,0.2,209.5800,396.92,1,399.200,467.858939,229.858001,...,2014,7,4,4,3,185,27,0,0,0
3,3,240,0,0.0,327.5922,394.57,0,419.990,467.858939,229.858001,...,2014,11,23,6,4,327,47,0,0,0
4,4,241,3,0.0,946.6800,393.62,1,514.500,416.248905,194.190000,...,2013,1,27,6,1,27,4,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50805,50805,51285,1,0.0,4.5000,0.01,3,13.020,121.097120,403.150068,...,2014,6,19,3,2,170,25,0,0,0
50806,50806,51286,3,0.8,-1.1100,0.01,3,0.444,121.097120,229.858001,...,2014,6,24,1,2,175,26,0,0,0
50807,50807,51287,1,0.0,11.2308,0.01,1,7.640,121.097120,229.858001,...,2013,12,2,0,4,336,49,0,0,0
50808,50808,51288,3,0.0,2.4000,0.00,3,6.720,121.097120,225.832657,...,2012,2,22,2,1,53,8,0,0,0


## Train test split

In [44]:
numerical = ['Discount',
             'Profit',
             'Shipping Cost',
             'Avg_Sales_Category',
             'Avg_Sales_Country',
             'Avg_Sales_Market',
             'Avg_Sales_Region',
             'Discount_value']

In [45]:
X = encoded_scaled_df.drop(columns=["Price"])
y = encoded_scaled_df["Price"]

In [46]:
scaler = StandardScaler()
X[numerical] = scaler.fit_transform(X[numerical])

y_scaler = StandardScaler()
y = y_scaler.fit_transform(y.values.reshape(-1, 1)).flatten()

In [47]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Models

In [48]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression, BayesianRidge, HuberRegressor, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

## Evaluation

In [54]:
def evaluate_model(model_name, y_test, y_pred, scaler):
    y_test_original = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    y_pred_original = scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()
    mae_ridge_original = mean_absolute_error(y_test_original, y_pred_original)
    mse_ridge_original = mean_squared_error(y_test_original, y_pred_original)
    rmse_ridge_original = mean_squared_error(y_test_original, y_pred_original, squared=False)
    r2_ridge_original = r2_score(y_test_original, y_pred_original)

    print(model_name)
    print(f"MAE: {mae_ridge_original}")
    print(f"MSE: {mse_ridge_original}")
    print(f"RMSE: {rmse_ridge_original}")
    print(f"R² Score: {r2_ridge_original}")

## Ridge Regression

In [55]:
param_grid_ridge = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
}

grid_ridge = GridSearchCV(
    estimator=Ridge(),
    param_grid=param_grid_ridge,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

model_ridge_tuned = grid_ridge.fit(X_train, y_train)
print(f"Ridge Best Parameters: {model_ridge_tuned.best_params_}")

y_pred_ridge_tuned = model_ridge_tuned.predict(X_test)
evaluate_model("Ridge Regression (Tuned)", y_test, y_pred_ridge_tuned, y_scaler)

Ridge Best Parameters: {'alpha': 100.0}
Ridge Regression (Tuned)
MAE: 32.66795520475548
MSE: 3361.6636639311864
RMSE: 57.97985567359741
R² Score: 0.6475334317166828


/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


## Lasso

In [56]:
param_grid_lasso = {
    'alpha': [0.01, 0.1, 1.0, 10.0]
}

grid_lasso = GridSearchCV(
    estimator=Lasso(),
    param_grid=param_grid_lasso,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

model_lasso_tuned = grid_lasso.fit(X_train, y_train)
print(f"Lasso Best Parameters: {model_lasso_tuned.best_params_}")

y_pred_lasso_tuned = model_lasso_tuned.predict(X_test)
evaluate_model("Lasso Regression (Tuned)", y_test, y_pred_lasso_tuned, y_scaler)

/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.347e+03, tolerance: 3.190e+00
  model = cd_fast.enet_coordinate_descent(
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.512e+03, tolerance: 3.268e+00
  model = cd_fast.enet_coordinate_descent(
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularis

Lasso Best Parameters: {'alpha': 0.01}
Lasso Regression (Tuned)
MAE: 34.008765742705414
MSE: 3668.3454525055495
RMSE: 60.56686761345306
R² Score: 0.6153781989569047


/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.039e+03, tolerance: 4.050e+00
  model = cd_fast.enet_coordinate_descent(
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


## Huber Regression

In [57]:
param_grid_huber = {
    'epsilon': [1.1, 1.35, 1.5],
    'alpha': [0.0001, 0.001, 0.01]
}

grid_huber = GridSearchCV(
    estimator=HuberRegressor(),
    param_grid=param_grid_huber,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

model_huber_tuned = grid_huber.fit(X_train, y_train)
print(f"Huber Best Parameters: {model_huber_tuned.best_params_}")

y_pred_huber_tuned = model_huber_tuned.predict(X_test)
evaluate_model("Robust Regression (Huber Tuned)", y_test, y_pred_huber_tuned, y_scaler)

/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED

Huber Best Parameters: {'alpha': 0.01, 'epsilon': 1.5}
Robust Regression (Huber Tuned)
MAE: 38.20385382578436
MSE: 4886.9923638173805
RMSE: 69.90702656970457
R² Score: 0.48760447210023206


/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
